# Multimodal Visual RAG: Qwen3-VL-2B generation (Colab)
Controlled against the completed SmolVLM-500M study: same 60 answerable, 60 withheld-evidence and 30 out-of-domain questions, same frozen retrieved pages, same evidence settings. Only the generator differs (`Qwen/Qwen3-VL-2B-Instruct@89644892`).

Select **Runtime → Change runtime type → T4 GPU**. Run cells 1–6 in order in every session; they verify the source bundle and the corpus.
The two GPU stages are gated: cell 8 (memory check) and cell 10 (full run) stop unless their approval flag is set to `True` by the owner.
After a disconnect, reconnect, rerun cells 1–6, then rerun the stage that was interrupted; completed answers are never regenerated.

Before the first session, put these in Google Drive (see `docs/COLAB_HANDOFF.md`):
- `MyDrive/mmrag/source/mmrag-<commit>.zip`
- `MyDrive/mmrag/corpus/mmrag_corpus-vidore_v3_computer_science-<fingerprint>.tar`

Do not paste tokens or credentials into this notebook.

In [ ]:
# 1. GPU check. Stop if this fails; no CPU or MPS fallback is supported.
import platform, subprocess
print(platform.python_version())
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# 2. Persistent storage. Outputs live on Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/mmrag')
OUTPUT = DRIVE / 'colab_outputs' / 'qwen3vl_2b_generation'
MEMORY_OUTPUT = DRIVE / 'colab_outputs' / 'qwen3vl_2b_memory_check'
for path in (OUTPUT, MEMORY_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
probe = MEMORY_OUTPUT / 'storage_probe.txt'
probe.write_text('persistent storage read-back')
assert probe.read_text() == 'persistent storage read-back'

In [ ]:
# 3. Source bundle from Drive, checked against values copied from the local manifest.
SOURCE_COMMIT = ''  # full 40-character SHA from colab_outputs/source/manifest.json
SOURCE_SHA256 = ''  # sha256 from the same manifest
assert len(SOURCE_COMMIT) == 40 and len(SOURCE_SHA256) == 64
import hashlib, io, zipfile
bundle = DRIVE / 'source' / f'mmrag-{SOURCE_COMMIT}.zip'
payload = bundle.read_bytes()
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256, 'Source checksum mismatch'
SOURCE = Path('/content') / f'mmrag-{SOURCE_COMMIT}'
if not SOURCE.exists():
    SOURCE.mkdir()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for item in archive.infolist():
            target = (SOURCE / item.filename).resolve()
            assert target.is_relative_to(SOURCE.resolve()), 'Unsafe archive path'
            assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks prohibited'
        archive.extractall(SOURCE)
assert (SOURCE / 'uv.lock').exists() and (SOURCE / 'src/mmrag/colab_generation.py').exists()
print('source', SOURCE_COMMIT, 'verified')

In [ ]:
# 4. Locked environment (same installer and extras as the passing smoke test).
import sys, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.11.16'], check=True)
UV = shutil.which('uv')
subprocess.run([UV, 'sync', '--frozen', '--python', '3.12', '--extra', 'models'], cwd=SOURCE, check=True)
PYTHON = str(SOURCE / '.venv/bin/python')
subprocess.run([PYTHON, '-c',
    'import torch, torchvision, transformers; '
    'print(torch.__version__, torchvision.__version__, transformers.__version__, torch.version.cuda); '
    'assert torch.cuda.is_available()'], check=True)

In [ ]:
# 5. Corpus: copy the archive to local disk, check its SHA-256, extract safely, verify every file.
CORPUS_ARCHIVE = ''  # file name from colab_outputs/corpus/*.json ('archive')
CORPUS_SHA256 = ''  # 'sha256' from the same file
assert CORPUS_ARCHIVE.endswith('.tar') and len(CORPUS_SHA256) == 64
import tarfile
local_tar = Path('/content') / CORPUS_ARCHIVE
if not local_tar.exists():
    shutil.copyfile(DRIVE / 'corpus' / CORPUS_ARCHIVE, local_tar)
digest = hashlib.sha256()
with local_tar.open('rb') as f:
    for block in iter(lambda: f.read(1 << 20), b''):
        digest.update(block)
assert digest.hexdigest() == CORPUS_SHA256, 'Corpus archive checksum mismatch'
EXTRACT = Path('/content/corpus_extract')
if not EXTRACT.exists():
    with tarfile.open(local_tar) as tar:
        tar.extractall(EXTRACT, filter='data')  # rejects absolute paths, links and traversal
CORPUS_ROOT = EXTRACT / 'mmrag_corpus'
subprocess.run([PYTHON, '-m', 'mmrag.colab_generation', 'verify-corpus',
                '--corpus-root', str(CORPUS_ROOT)], cwd=SOURCE, check=True)

In [ ]:
# 6. Helpers: stream a stage to the screen and an append-only log; verify checksum files.
import json
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a') as log:
        log.write(f'\n=== {" ".join(command[2:5])} ===\n')
        process = subprocess.Popen(command, cwd=SOURCE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        return process.wait()

def sha256_path(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def verify_checksums(root):
    lines = (root / 'CHECKSUMS.sha256').read_text().splitlines()
    for line in lines:
        digest, name = line.split('  ', 1)
        assert sha256_path(root / name) == digest, f'Checksum mismatch: {name}'
    print(len(lines), 'files verified in', root)
print('ready: source', SOURCE_COMMIT[:7], '| corpus', CORPUS_ROOT)

## Memory validation (requires owner approval)
Runs the five heaviest real prompts (three 1024-px pages plus their OCR text, `image_text_rag`), then the heaviest image-only and text-only prompt, once each. Records input sizes, tokens, latency and peak allocated/reserved VRAM per prompt. Passes only if peak reserved VRAM stays below 80% of the GPU total. Writes only to `qwen3vl_2b_memory_check/`; never touches experiment outputs.

In [ ]:
# 8. Memory check. Set to True only after the owner approves this GPU run.
APPROVE_MEMORY_CHECK = False
assert APPROVE_MEMORY_CHECK, 'Memory check not approved: stop here'
rc = run_logged([PYTHON, '-m', 'mmrag.colab_generation', 'memory-check',
                 '--corpus-root', str(CORPUS_ROOT), '--output', str(MEMORY_OUTPUT),
                 '--source-commit', SOURCE_COMMIT], MEMORY_OUTPUT / 'console.log')
report_path = sorted(MEMORY_OUTPUT.glob('memory_check/*/report.json'), key=lambda p: p.stat().st_mtime)[-1]
assert sha256_path(report_path) == (report_path.parent / 'report.sha256').read_text().split()[0]
report = json.loads(report_path.read_text())
GiB = 1024 ** 3
for e in report['examples']:
    print(f"{e['system']:28s} images={e['image_sizes_after_resize']} in={e['input_tokens']} "
          f"out={e['output_tokens']} {e['latency_s']:.2f}s "
          f"alloc={e.get('peak_allocated_bytes', 0) / GiB:.2f} GiB reserved={e.get('peak_reserved_bytes', 0) / GiB:.2f} GiB")
print('status:', report['status'], '| errors:', report['errors'])
assert rc == 0 and report['status'] == 'passed', 'Memory check failed; keep the report and stop'

## Full experiment (requires separate owner approval after the memory check)
540 generations: closed book 90, text 150, image 150, image + text 150. Answers are appended to `qwen3vl_2b_generation/runs/*.jsonl` one at a time. If the runtime disconnects, rerun cells 1–6 and this cell: completed answers are skipped, and a half-written last line is moved aside to a `.partial-*` file. The output directory is bound to this source commit and configuration; a different commit is refused. Scoring happens afterwards on the local machine, not here.

In [ ]:
# 10. Full run (resumable). Set to True only after the owner approves the full experiment.
APPROVE_FULL_RUN = False
assert APPROVE_FULL_RUN, 'Full run not approved: stop here'
rc = run_logged([PYTHON, '-m', 'mmrag.colab_generation', 'run',
                 '--corpus-root', str(CORPUS_ROOT), '--output', str(OUTPUT),
                 '--source-commit', SOURCE_COMMIT], OUTPUT / 'logs' / 'console.log')
print('exit status:', rc)
verify_checksums(OUTPUT)
for done in sorted((OUTPUT / 'runs').glob('*.complete.json')):
    d = json.loads(done.read_text())
    print(d['system'], d['n'], 'records, peak reserved', round(d['peak_reserved_bytes'] / 1024 ** 3, 2), 'GiB')
assert rc == 0, 'Run stopped early; rerun cells 1-6 and this cell to resume'

## Return the evidence
Download the whole `MyDrive/mmrag/colab_outputs/qwen3vl_2b_generation/` folder (and `qwen3vl_2b_memory_check/` after cell 8). Keep failed or partial outputs as they are. Do not start other models from this notebook.